<a href="https://colab.research.google.com/github/CosbyCD/Current-Projects/blob/main/Cyclistic-Phase2/Cyclistic_Phase2_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install clickhouse-connect -q

In [5]:
import io
import zipfile
import requests
import pandas as pd
import clickhouse_connect

In [6]:
client = clickhouse_connect.get_client(
    host="fu5itnlxt3.us-west-2.aws.clickhouse.cloud",
    port=8443,
    username="default",
    password="your_password_here",
    secure=True
)

print(client.server_version)

25.12.1.1606


In [7]:
month = "202201"
url = f"https://divvy-tripdata.s3.amazonaws.com/{month}-divvy-tripdata.zip"

print(f"Downloading {url}...")
response = requests.get(url, timeout=60)
response.raise_for_status()
print(f"Download OK — {len(response.content):,} bytes")

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    csv_name = [f for f in z.namelist() if f.endswith(".csv")][0]
    print(f"CSV file inside ZIP: {csv_name}")
    with z.open(csv_name) as f:
        df = pd.read_csv(f, dtype=str)

print(f"\nRows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst row:")
print(df.iloc[0].to_dict())

Download OK — 3,837,661 bytes
CSV file inside ZIP: 202201-divvy-tripdata.csv

Rows: 103,770
Columns: ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']

First row:
{'ride_id': 'C2F7DD78E82EC875', 'rideable_type': 'electric_bike', 'started_at': '2022-01-13 11:59:47', 'ended_at': '2022-01-13 12:02:44', 'start_station_name': 'Glenwood Ave & Touhy Ave', 'start_station_id': '525', 'end_station_name': 'Clark St & Touhy Ave', 'end_station_id': 'RP-007', 'start_lat': '42.0128005', 'start_lng': '-87.665906', 'end_lat': '42.01256011541', 'end_lng': '-87.6743671152', 'member_casual': 'casual'}


In [8]:
# Parse datetimes
df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"]   = pd.to_datetime(df["ended_at"])

# Parse coordinates
df["start_lat"] = pd.to_numeric(df["start_lat"], errors="coerce")
df["start_lng"] = pd.to_numeric(df["start_lng"], errors="coerce")
df["end_lat"]   = pd.to_numeric(df["end_lat"],   errors="coerce")
df["end_lng"]   = pd.to_numeric(df["end_lng"],   errors="coerce")

# Drop null coordinates
before = len(df)
df = df.dropna(subset=["start_lat", "start_lng", "end_lat", "end_lng"])
print(f"Rows dropped for null coordinates: {before - len(df):,}")
print(f"Rows remaining: {len(df):,}")

# Station IDs and names
df["start_station_id"]   = df["start_station_id"].fillna("").astype(str)
df["end_station_id"]     = df["end_station_id"].fillna("").astype(str)
df["start_station_name"] = df["start_station_name"].fillna("")
df["end_station_name"]   = df["end_station_name"].fillna("")

# Derived columns
df["ride_duration_min"] = (
    (df["ended_at"] - df["started_at"])
    .dt.total_seconds()
    .div(60)
    .clip(lower=0)
    .astype("int32")
)
df["ride_date"] = pd.to_datetime(df["started_at"].dt.date)
df["ride_hour"] = df["started_at"].dt.hour.astype("int8")

print(f"\nSample derived values:")
print(df[["ride_id","ride_duration_min","ride_date","ride_hour"]].head(3))

Rows dropped for null coordinates: 86
Rows remaining: 103,684

Sample derived values:
            ride_id  ride_duration_min  ride_date  ride_hour
0  C2F7DD78E82EC875                  2 2022-01-13         11
1  A6CF8980A652D272                  4 2022-01-10          8
2  BD0F91DFF741C66D                  4 2022-01-25          4


In [9]:
columns = [
    "ride_id", "rideable_type", "started_at", "ended_at",
    "start_station_name", "start_station_id",
    "end_station_name", "end_station_id",
    "start_lat", "start_lng", "end_lat", "end_lng",
    "member_casual", "ride_duration_min", "ride_date", "ride_hour"
]

client.insert_df("cyclistic_rides", df[columns])
print(f"Inserted {len(df):,} rows into cyclistic_rides")

Inserted 103,684 rows into cyclistic_rides


In [10]:
MONTHS = [f"2022{str(m).zfill(2)}" for m in range(2, 13)]
BASE_URL = "https://divvy-tripdata.s3.amazonaws.com/"
total = 103684  # January already loaded

for month in MONTHS:
    print(f"[{month}]")

    url = f"{BASE_URL}{month}-divvy-tripdata.zip"
    print(f"  Downloading...")
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        csv_name = [f for f in z.namelist() if f.endswith(".csv")][0]
        with z.open(csv_name) as f:
            df = pd.read_csv(f, dtype=str)

    print(f"  Extracted {len(df):,} rows")

    df["started_at"] = pd.to_datetime(df["started_at"])
    df["ended_at"]   = pd.to_datetime(df["ended_at"])
    df["start_lat"]  = pd.to_numeric(df["start_lat"], errors="coerce")
    df["start_lng"]  = pd.to_numeric(df["start_lng"], errors="coerce")
    df["end_lat"]    = pd.to_numeric(df["end_lat"],   errors="coerce")
    df["end_lng"]    = pd.to_numeric(df["end_lng"],   errors="coerce")

    before = len(df)
    df = df.dropna(subset=["start_lat", "start_lng", "end_lat", "end_lng"])
    dropped = before - len(df)
    if dropped > 0:
        print(f"  Dropped {dropped:,} null coordinate rows")

    df["start_station_id"]   = df["start_station_id"].fillna("").astype(str)
    df["end_station_id"]     = df["end_station_id"].fillna("").astype(str)
    df["start_station_name"] = df["start_station_name"].fillna("")
    df["end_station_name"]   = df["end_station_name"].fillna("")

    df["ride_duration_min"] = (
        (df["ended_at"] - df["started_at"])
        .dt.total_seconds()
        .div(60)
        .clip(lower=0)
        .astype("int32")
    )
    df["ride_date"] = pd.to_datetime(df["started_at"].dt.date)
    df["ride_hour"] = df["started_at"].dt.hour.astype("int8")

    columns = [
        "ride_id", "rideable_type", "started_at", "ended_at",
        "start_station_name", "start_station_id",
        "end_station_name", "end_station_id",
        "start_lat", "start_lng", "end_lat", "end_lng",
        "member_casual", "ride_duration_min", "ride_date", "ride_hour"
    ]

    client.insert_df("cyclistic_rides", df[columns])
    total += len(df)
    print(f"  Inserted {len(df):,} rows — running total: {total:,}\n")

print(f"Pipeline complete — {total:,} total rows loaded")

[202202]
  Downloading...
  Extracted 115,609 rows
  Dropped 77 null coordinate rows
  Inserted 115,532 rows — running total: 219,216

[202203]
  Downloading...
  Extracted 284,042 rows
  Dropped 266 null coordinate rows
  Inserted 283,776 rows — running total: 502,992

[202204]
  Downloading...
  Extracted 371,249 rows
  Dropped 317 null coordinate rows
  Inserted 370,932 rows — running total: 873,924

[202205]
  Downloading...
  Extracted 634,858 rows
  Dropped 722 null coordinate rows
  Inserted 634,136 rows — running total: 1,508,060

[202206]
  Downloading...
  Extracted 769,204 rows
  Dropped 1,055 null coordinate rows
  Inserted 768,149 rows — running total: 2,276,209

[202207]
  Downloading...
  Extracted 823,488 rows
  Dropped 947 null coordinate rows
  Inserted 822,541 rows — running total: 3,098,750

[202208]
  Downloading...
  Extracted 785,932 rows
  Dropped 843 null coordinate rows
  Inserted 785,089 rows — running total: 3,883,839

[202209]
  Downloading...
  Extracted 7

In [14]:
weather_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/retrievebulkdataset?&key=8YAPFCXSKYSB93HVNVPZXDLAR&taskId=b1f1ca527fb0563cf7ee5d58c48a2ec6&zip=false"

response = requests.get(weather_url)
response.raise_for_status()

weather_df = pd.read_csv(io.StringIO(response.text))
print(f"Rows: {len(weather_df)}")
print(f"Columns: {list(weather_df.columns)}")
print(weather_df.head(2))

Rows: 365
Columns: ['name', 'datetime', 'tempmax', 'tempmin', 'temp', 'feelslike', 'humidity', 'precip', 'precipprob', 'preciptype', 'snow', 'snowdepth', 'windgust', 'windspeed', 'windspeedmax', 'windspeedmean', 'windspeedmin', 'winddir', 'cloudcover', 'visibility', 'uvindex', 'uvindex2', 'severerisk', 'lightningrisk', 'hailrisk', 'hailsize', 'hailprobability', 'lightningdensity', 'sunrise', 'sunset', 'moonphase', 'moonrise', 'moonset', 'o3', 'aqius', 'conditions', 'description', 'icon', 'stations', 'source']
                                                name    datetime  tempmax  \
0   Chicago, Illinois Date range: 2022-01-01 to 2...  2022-01-01     41.5   
1   Chicago, Illinois Date range: 2022-01-01 to 2...  2022-01-02     30.4   

   tempmin  temp  feelslike  humidity  precip  precipprob preciptype  ...  \
0     30.5  35.3       24.9      81.4   0.057         100  rain,snow  ...   
1     16.3  24.0       13.5      73.3   0.030         100  rain,snow  ...   

                     

In [15]:
print(weather_df.columns.tolist())
print()
print(weather_df[['datetime','tempmax','tempmin','temp','moonphase','moonrise','moonset','sunrise','sunset','conditions']].head(3))

['name', 'datetime', 'tempmax', 'tempmin', 'temp', 'feelslike', 'humidity', 'precip', 'precipprob', 'preciptype', 'snow', 'snowdepth', 'windgust', 'windspeed', 'windspeedmax', 'windspeedmean', 'windspeedmin', 'winddir', 'cloudcover', 'visibility', 'uvindex', 'uvindex2', 'severerisk', 'lightningrisk', 'hailrisk', 'hailsize', 'hailprobability', 'lightningdensity', 'sunrise', 'sunset', 'moonphase', 'moonrise', 'moonset', 'o3', 'aqius', 'conditions', 'description', 'icon', 'stations', 'source']

     datetime  tempmax  tempmin  temp                     moonphase  \
0  2022-01-01     41.5     30.5  35.3  Snow, Rain, Partially cloudy   
1  2022-01-02     30.4     16.3  24.0  Snow, Rain, Partially cloudy   
2  2022-01-03     23.6     10.4  17.2                         Clear   

                                            moonrise    moonset  sunrise  \
0  Partly cloudy throughout the day with rain or ...       snow      NaN   
1  Partly cloudy throughout the day with early mo...       snow   

In [16]:
weather_df = pd.read_csv('/content/Cyclistin Phase2 - Chicago, Illinois Date range_ 2022-01-01 to 2022-12-31.csv')
print(f"Rows: {len(weather_df)}")
print(f"Columns: {list(weather_df.columns)}")
print(weather_df.head(2))

Rows: 365
Columns: ['name', 'datetime', 'tempmax', 'tempmin', 'temp', 'feelslike', 'humidity', 'precip', 'precipprob', 'preciptype', 'snow', 'snowdepth', 'windgust', 'windspeed', 'windspeedmax', 'windspeedmean', 'windspeedmin', 'winddir', 'cloudcover', 'visibility', 'uvindex', 'uvindex2', 'severerisk', 'lightningrisk', 'hailrisk', 'hailsize', 'hailprobability', 'lightningdensity', 'sunrise', 'sunset', 'moonphase', 'moonrise', 'moonset', 'o3', 'aqius', 'conditions', 'description', 'icon', 'stations', 'source']
                                                name    datetime  tempmax  \
0   Chicago, Illinois Date range: 2022-01-01 to 2...  2022-01-01     41.5   
1   Chicago, Illinois Date range: 2022-01-01 to 2...  2022-01-02     30.4   

   tempmin  temp  feelslike  humidity  precip  precipprob preciptype  ...  \
0     30.5  35.3       24.9      81.4   0.057         100  rain,snow  ...   
1     16.3  24.0       13.5      73.3   0.030         100  rain,snow  ...   

                     

In [17]:
with open('/content/Cyclistin Phase2 - Chicago, Illinois Date range_ 2022-01-01 to 2022-12-31.csv', 'r') as f:
    for i, line in enumerate(f):
        print(line)
        if i > 3:
            break

name,datetime,tempmax,tempmin,temp,feelslike,humidity,precip,precipprob,preciptype,snow,snowdepth,windgust,windspeed,windspeedmax,windspeedmean,windspeedmin,winddir,cloudcover,visibility,uvindex,uvindex2,severerisk,lightningrisk,hailrisk,hailsize,hailprobability,lightningdensity,sunrise,sunset,moonphase,moonrise,moonset,o3,aqius,conditions,description,icon,stations,source

" Chicago, Illinois Date range: 2022-01-01 to 2022-12-31",2022-01-01,41.5,30.5,35.3,24.9,81.4,0.057,100,"rain,snow",1.2,0.4,35,23.3,23.3,18.3,10.4,24.9,71.9,4.7,0,,,2022-01-01T07:19:06,2022-01-01T16:31:24,0.96,2022-01-01T06:17:33,2022-01-01T15:16:50,,,"Snow, Rain, Partially cloudy",Partly cloudy throughout the day with rain or snow.,snow,"72534014819,USW00014819,C8740,KORD,KLOT,KMDW,72530094846,F7086",obs

" Chicago, Illinois Date range: 2022-01-01 to 2022-12-31",2022-01-02,30.4,16.3,24,13.5,73.3,0.03,100,"rain,snow",0.9,1.8,26.1,19.8,19.8,11.2,6.5,337.9,49.6,7.8,4,,,2022-01-02T07:19:10,2022-01-02T16:32:17,0,2022-01-

In [18]:
weather_df = pd.read_csv('/content/Cyclistin Phase2 - Chicago, Illinois Date range_ 2022-01-01 to 2022-12-31.csv')

# Rename scrambled columns to what they actually contain
weather_df = weather_df.rename(columns={
    'severerisk':     'sunrise',
    'lightningrisk':  'sunset',
    'hailrisk':       'moonphase',
    'hailsize':       'moonrise',
    'hailprobability':'moonset',
    'o3':             'conditions_text',
    'aqius':          'description_text',
    'conditions':     'icon_text',
    'description':    'conditions',
    'icon':           'description',
    'stations':       'icon',
    'source':         'stations'
})

print(weather_df[['datetime','tempmax','tempmin','temp','sunrise','sunset','moonphase','moonrise','moonset','conditions']].head(3))

     datetime  tempmax  tempmin  temp  sunrise  sunrise               sunset  \
0  2022-01-01     41.5     30.5  35.3      NaN      NaN  2022-01-01T07:19:06   
1  2022-01-02     30.4     16.3  24.0      NaN      NaN  2022-01-02T07:19:10   
2  2022-01-03     23.6     10.4  17.2      NaN      NaN  2022-01-03T07:19:12   

   sunset            moonphase                     moonphase moonrise  \
0     NaN  2022-01-01T16:31:24  Snow, Rain, Partially cloudy     0.96   
1     NaN  2022-01-02T16:32:17  Snow, Rain, Partially cloudy     0.00   
2     NaN  2022-01-03T16:33:11                         Clear     0.03   

                                            moonrise              moonset  \
0  Partly cloudy throughout the day with rain or ...  2022-01-01T06:17:33   
1  Partly cloudy throughout the day with early mo...  2022-01-02T07:29:59   
2               Clear conditions throughout the day.  2022-01-03T08:29:59   

     moonset  conditions  
0       snow         NaN  
1       snow         Na

In [19]:
import pandas as pd

# Read raw without header interpretation
raw = pd.read_csv('/content/Cyclistin Phase2 - Chicago, Illinois Date range_ 2022-01-01 to 2022-12-31.csv', header=None, skiprows=1)

# Assign only the columns we actually want by position
weather_df = pd.DataFrame({
    'datetime':     raw[1],
    'tempmax':      raw[2],
    'tempmin':      raw[3],
    'temp':         raw[4],
    'feelslike':    raw[5],
    'humidity':     raw[6],
    'precip':       raw[7],
    'precipprob':   raw[8],
    'preciptype':   raw[9],
    'snow':         raw[10],
    'snowdepth':    raw[11],
    'windspeed':    raw[13],
    'cloudcover':   raw[18],
    'visibility':   raw[19],
    'uvindex':      raw[20],
    'sunrise':      raw[22],
    'sunset':       raw[23],
    'moonphase':    raw[24],
    'moonrise':     raw[25],
    'moonset':      raw[26],
    'conditions':   raw[29]
})

print(f"Rows: {len(weather_df)}")
print(weather_df.head(3))

Rows: 365
     datetime  tempmax  tempmin  temp  feelslike  humidity  precip  \
0  2022-01-01     41.5     30.5  35.3       24.9      81.4   0.057   
1  2022-01-02     30.4     16.3  24.0       13.5      73.3   0.030   
2  2022-01-03     23.6     10.4  17.2        6.9      62.1   0.000   

   precipprob preciptype  snow  ...  windspeed  cloudcover  visibility  \
0         100  rain,snow   1.2  ...       23.3        71.9         4.7   
1         100  rain,snow   0.9  ...       19.8        49.6         7.8   
2           0        NaN   0.0  ...       13.4         8.9         9.9   

   uvindex  sunrise               sunset            moonphase moonrise  \
0        0      NaN  2022-01-01T07:19:06  2022-01-01T16:31:24     0.96   
1        4      NaN  2022-01-02T07:19:10  2022-01-02T16:32:17     0.00   
2        4      NaN  2022-01-03T07:19:12  2022-01-03T16:33:11     0.03   

               moonset conditions  
0  2022-01-01T06:17:33        NaN  
1  2022-01-02T07:29:59        NaN  
2  2022

In [20]:
print(raw.iloc[0].to_dict())

{0: ' Chicago, Illinois Date range: 2022-01-01 to 2022-12-31', 1: '2022-01-01', 2: 41.5, 3: 30.5, 4: 35.3, 5: 24.9, 6: 81.4, 7: 0.057, 8: 100, 9: 'rain,snow', 10: 1.2, 11: 0.4, 12: 35.0, 13: 23.3, 14: 23.3, 15: 18.3, 16: 10.4, 17: 24.9, 18: 71.9, 19: 4.7, 20: 0, 21: nan, 22: nan, 23: '2022-01-01T07:19:06', 24: '2022-01-01T16:31:24', 25: 0.96, 26: '2022-01-01T06:17:33', 27: '2022-01-01T15:16:50', 28: nan, 29: nan, 30: 'Snow, Rain, Partially cloudy', 31: 'Partly cloudy throughout the day with rain or snow.', 32: 'snow', 33: '72534014819,USW00014819,C8740,KORD,KLOT,KMDW,72530094846,F7086', 34: 'obs'}


In [21]:
weather_df = pd.DataFrame({
    'datetime':     raw[1],
    'tempmax':      raw[2],
    'tempmin':      raw[3],
    'temp':         raw[4],
    'feelslike':    raw[5],
    'humidity':     raw[6],
    'precip':       raw[7],
    'precipprob':   raw[8],
    'preciptype':   raw[9],
    'snow':         raw[10],
    'snowdepth':    raw[11],
    'windspeed':    raw[13],
    'cloudcover':   raw[18],
    'visibility':   raw[19],
    'uvindex':      raw[20],
    'sunrise':      raw[23],
    'sunset':       raw[24],
    'moonphase':    raw[25],
    'moonrise':     raw[26],
    'moonset':      raw[27],
    'conditions':   raw[30]
})

print(f"Rows: {len(weather_df)}")
print(weather_df[['datetime','tempmax','sunrise','sunset','moonphase','moonrise','moonset','conditions']].head(3))

Rows: 365
     datetime  tempmax              sunrise               sunset  moonphase  \
0  2022-01-01     41.5  2022-01-01T07:19:06  2022-01-01T16:31:24       0.96   
1  2022-01-02     30.4  2022-01-02T07:19:10  2022-01-02T16:32:17       0.00   
2  2022-01-03     23.6  2022-01-03T07:19:12  2022-01-03T16:33:11       0.03   

              moonrise              moonset                    conditions  
0  2022-01-01T06:17:33  2022-01-01T15:16:50  Snow, Rain, Partially cloudy  
1  2022-01-02T07:29:59  2022-01-02T16:21:22  Snow, Rain, Partially cloudy  
2  2022-01-03T08:29:59  2022-01-03T17:35:53                         Clear  


In [22]:
print({i: raw.iloc[0][i] for i in range(35)})

{0: ' Chicago, Illinois Date range: 2022-01-01 to 2022-12-31', 1: '2022-01-01', 2: np.float64(41.5), 3: np.float64(30.5), 4: np.float64(35.3), 5: np.float64(24.9), 6: np.float64(81.4), 7: np.float64(0.057), 8: np.int64(100), 9: 'rain,snow', 10: np.float64(1.2), 11: np.float64(0.4), 12: np.float64(35.0), 13: np.float64(23.3), 14: np.float64(23.3), 15: np.float64(18.3), 16: np.float64(10.4), 17: np.float64(24.9), 18: np.float64(71.9), 19: np.float64(4.7), 20: np.int64(0), 21: np.float64(nan), 22: np.float64(nan), 23: '2022-01-01T07:19:06', 24: '2022-01-01T16:31:24', 25: np.float64(0.96), 26: '2022-01-01T06:17:33', 27: '2022-01-01T15:16:50', 28: np.float64(nan), 29: np.float64(nan), 30: 'Snow, Rain, Partially cloudy', 31: 'Partly cloudy throughout the day with rain or snow.', 32: 'snow', 33: '72534014819,USW00014819,C8740,KORD,KLOT,KMDW,72530094846,F7086', 34: 'obs'}


In [24]:
weather_df = pd.DataFrame({
    'datetime':       raw[1],
    'tempmax':        raw[2],
    'tempmin':        raw[3],
    'temp':           raw[4],
    'feelslike':      raw[5],
    'humidity':       raw[6],
    'precip':         raw[7],
    'precipprob':     raw[8],
    'preciptype':     raw[9],
    'snow':           raw[10],
    'snowdepth':      raw[11],
    'windgust':       raw[12],
    'windspeed':      raw[13],
    'windspeedmax':   raw[14],
    'windspeedmean':  raw[15],
    'windspeedmin':   raw[16],
    'winddir':        raw[17],
    'cloudcover':     raw[18],
    'visibility':     raw[19],
    'uvindex':        raw[20],
    'solarradiation': raw[21],
    'solarenergy':    raw[22],
    'sunrise':        raw[23],
    'sunset':         raw[24],
    'moonphase':      raw[25],
    'moonrise':       raw[26],
    'moonset':        raw[27],
    'conditions':     raw[30]
})

print(f"Rows: {len(weather_df)}")
print(weather_df[['datetime','windgust','windspeed','winddir','solarradiation','conditions']].head(3))

Rows: 365
     datetime  windgust  windspeed  winddir  solarradiation  \
0  2022-01-01      35.0       23.3     24.9             NaN   
1  2022-01-02      26.1       19.8    337.9             NaN   
2  2022-01-03      25.3       13.4    228.1             NaN   

                     conditions  
0  Snow, Rain, Partially cloudy  
1  Snow, Rain, Partially cloudy  
2                         Clear  


In [27]:
# Parse datetime to date
weather_df['datetime'] = pd.to_datetime(weather_df['datetime']).dt.date

# Fill nulls
float_cols = ['tempmax','tempmin','temp','feelslike','humidity','precip','precipprob',
              'snow','snowdepth','windgust','windspeed','windspeedmax','windspeedmean',
              'windspeedmin','winddir','cloudcover','visibility','solarradiation','solarenergy',
              'moonphase']
for col in float_cols:
    weather_df[col] = pd.to_numeric(weather_df[col], errors='coerce').fillna(0.0).astype('float32')

weather_df['uvindex'] = pd.to_numeric(weather_df['uvindex'], errors='coerce').fillna(0).astype('uint8')
weather_df['preciptype'] = weather_df['preciptype'].fillna('').astype(str)
weather_df['conditions'] = weather_df['conditions'].fillna('').astype(str)
weather_df['sunrise'] = weather_df['sunrise'].fillna('').astype(str)
weather_df['sunset'] = weather_df['sunset'].fillna('').astype(str)
weather_df['moonrise'] = weather_df['moonrise'].fillna('').astype(str)
weather_df['moonset'] = weather_df['moonset'].fillna('').astype(str)

# Rename datetime to weather_date
weather_df = weather_df.rename(columns={'datetime': 'weather_date'})

# Insert into ClickHouse
client.insert_df('weather_data', weather_df)
print(f"Inserted {len(weather_df)} rows into weather_data")

Inserted 365 rows into weather_data


In [28]:
holidays_data = [
    ("New Year's", "2022-01-01"),
    ("Martin Luther King Jr.", "2022-01-17"),
    ("Presidents Day", "2022-02-21"),
    ("Easter", "2022-03-23"),
    ("Memorial Day", "2022-05-27"),
    ("Independence Day", "2022-07-04"),
    ("Columbus Day", "2022-10-04"),
    ("Halloween", "2022-10-31"),
    ("Veterans Day", "2022-11-11"),
    ("Thanksgiving", "2022-11-24"),
    ("Christmas Day", "2022-12-25"),
    ("Mother's Day", "2022-05-18"),
    ("Father's Day", "2022-06-16"),
    ("Boxing Day", "2022-12-26"),
    ("International Worker's Day (May Day)", "2022-05-01"),
    ("Flag Day", "2022-06-14"),
    ("Earth Day", "2022-04-22"),
    ("Pi Day", "2022-03-14"),
    ("International Women's Day", "2022-03-08"),
    ("Valentine's Day", "2022-02-14"),
    ("Juneteenth", "2022-06-19"),
    ("Chinese New Year", "2022-02-01"),
    ("Diwali", "2022-10-24"),
    ("Eid al-Fitr (End of Ramadan)", "2022-05-02"),
    ("Eid al-Adha", "2022-07-09"),
    ("Mardi Gras", "2022-03-01"),
    ("Chicago St. Patrick's Day Parade", "2022-03-12"),
    ("Chicago Pride Parade", "2022-06-26"),
    ("Oktoberfest", "2022-09-10"),
    ("Oktoberfest", "2022-09-11"),
    ("Oktoberfest", "2022-09-12"),
    ("Oktoberfest", "2022-09-13"),
    ("Oktoberfest", "2022-09-14"),
    ("Oktoberfest", "2022-09-15"),
    ("Oktoberfest", "2022-09-16"),
    ("Oktoberfest", "2022-09-17"),
    ("Oktoberfest", "2022-09-18"),
    ("Oktoberfest", "2022-09-19"),
    ("Chicago Air and Water Show", "2022-08-20"),
    ("Chicago Air and Water Show", "2022-08-21"),
    ("Taste of Chicago", "2022-06-08"),
    ("Taste of Chicago", "2022-06-09"),
    ("Taste of Chicago", "2022-06-10"),
    ("Chicago Blues Festival", "2022-06-09"),
    ("Chicago Blues Festival", "2022-06-10"),
    ("Chicago Blues Festival", "2022-06-11"),
    ("Chicago Blues Festival", "2022-06-12"),
    ("Chicago Jazz Festival", "2022-09-01"),
    ("Chicago Jazz Festival", "2022-09-02"),
    ("Chicago Jazz Festival", "2022-09-03"),
    ("Chicago Jazz Festival", "2022-09-04"),
    ("Lollapalooza", "2022-07-28"),
    ("Lollapalooza", "2022-07-29"),
    ("Lollapalooza", "2022-07-30"),
    ("Lollapalooza", "2022-07-31"),
]

import pandas as pd
holidays_df = pd.DataFrame(holidays_data, columns=["holiday_name", "holiday_date"])
holidays_df["holiday_date"] = pd.to_datetime(holidays_df["holiday_date"]).dt.date

client.insert_df("holidays", holidays_df)
print(f"Inserted {len(holidays_df)} rows into holidays")

Inserted 55 rows into holidays
